# LLM Layer-wise Entropy Analysis

Estimates H(Z), I(input_token; Z), and I(output_token; Z) for every layer of a small causal LM using `LLMInferrer` and `SoftEntropyAccumulator`.

Defaults are set to run on CPU in a few minutes using [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) and 500 examples from the [reddit](https://huggingface.co/datasets/sentence-transformers/reddit) dataset.

```bash
pip install soft-entropy[llm]
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from _conf import *

from soft_entropy import LLMInferrer

## Run inference

`LLMInferrer` loads the model and dataset, streams examples in batches, and accumulates soft entropy counts layer by layer. No representations are stored — memory usage is constant in the number of examples.

In [ ]:
inferrer = LLMInferrer(
    model_id="HuggingFaceTB/SmolLM2-135M-Instruct",
    dataset_id="sentence-transformers/reddit",
    label_types=["unigram", "bigram", "trigram"],
    max_length=64,
)

results = inferrer.run(n_examples=500, batch_size=4)

per_layer = results["per_layer"]  # list of dicts, index 0 = embedding layer
mean      = results["mean"]       # averaged across transformer layers

print(f"Layers analysed (incl. embedding): {len(per_layer)}")
print()
print("Mean across transformer layers:")
for k, v in mean.items():
    print(f"  {k:<40} {v:.4f}")

## Per-layer results table

In [ ]:
orders = [lt for lt in ["unigram", "bigram", "trigram"] if lt in inferrer.label_types]

def get(key):
    return [r.get(key, float("nan")) for r in per_layer]

h_vals = get("H(Z)")

# collect per-order series
mi_in  = {o: get(f"I(X;Z)/input_{o}")  for o in orders}
mi_out = {o: get(f"I(X;Z)/output_{o}") for o in orders}
optimality = {
    o: [out / inp if inp > 0 else float("nan")
        for inp, out in zip(mi_in[o], mi_out[o])]
    for o in orders
}

# --- table ---
header = f"  {'layer':>6}  {'H(Z)':>7}"
for o in orders:
    short = o[:3]
    header += f"  {'I(in;'+short+')':>10}  {'I(out;'+short+')':>11}  {'opt('+short+')':>9}"
print(header)
print("  " + "-" * (len(header) - 2))
for i in range(len(per_layer)):
    label = "embed" if i == 0 else str(i)
    row = f"  {label:>6}  {h_vals[i]:>7.4f}"
    for o in orders:
        row += f"  {mi_in[o][i]:>10.4f}  {mi_out[o][i]:>11.4f}  {optimality[o][i]:>9.4f}"
    print(row)

## Layer-wise entropy and mutual information

Four metrics tracked across every layer (index 0 = embedding, last = final transformer block):

- **H(Z)** — representation entropy. High entropy means the layer's activations are spread across many directions; it tends to drop with depth as the model focuses on task-relevant structure.
- **I(input n-gram; Z)** — mutual information between the representation and the input token's n-gram context. Measures how much of the raw input signal is still present.
- **I(output n-gram; Z)** — mutual information between the representation and the predicted output token's n-gram context. Tracks how much prediction-relevant structure has been built up.
- **Optimality** — ratio I(output; Z) / I(input; Z). Values near 1 mean the layer retains input information only insofar as it is useful for prediction — a sign of efficient compression.

![Layer-wise entropy and mutual information](figures/llm_entropy.png)

## Information plane

Each point is one layer; the trajectory runs from the embedding layer to the final transformer block. The x-axis measures compression — how much information the representation retains about the input token. The y-axis measures prediction — how much it captures about the output token. The dashed diagonal is the Information Bottleneck bound (optimality = 1): layers on the diagonal pass through only input information that is directly useful for prediction. Layers moving down and to the left across depth are compressing; layers moving up toward the diagonal are becoming more predictively efficient.

![Information plane](figures/llm_information_plane.png)